# Podstawy Pythona w analizie danych · Laboratorium 2
## Nieruchomości: od danych do rekomendacji

### Case study: wybór nieruchomości dla klienta

Pracujesz w zespole analiz rynku nieruchomości. Klient szuka nieruchomości w budynku mającym **nie więcej niż 20 lat**, położonej **do 800 metrów od stacji metra**, z **co najmniej 3 sklepami** w otoczeniu. Chce porównać ceny jednostkowe i poznać kompromis między dostępnością komunikacji a ceną.

Twoje zadanie: rozpoznać dane, wybrać rekordy spełniające kryteria, porównać segmenty rynku i przygotować **tabelę 5 kandydatów i krótką rekomendację**. Dane mają charakter historyczny, więc wynik to ćwiczenie analityczne, a nie lista aktualnych ofert.

Wszystkie bloki wracają do tych samych nieruchomości. Najpierw krótki przykład, następnie Twoja praca. Poznasz operatory, funkcje, tekst, regex, daty oraz pandas: filtry, nowe kolumny, `groupby`, agregacje i tabelę przestawną.

### Plan · 2 × 60 minut + jedna przerwa 10 minut

**120 minut pracy, 130 minut łącznie z przerwą.** Zadania rozszerzające nie wchodzą do tego czasu.

| Czas od początku | Część | Minuty pracy |
|---|---|---:|
| 0–10 | **Blok I:** wczytane dane i rozpoznanie | 10 |
| 10–25 | Operatory, jednostki i nowe kolumny | 15 |
| 25–45 | Filtry i lista kandydatów | 20 |
| 45–60 | Regex od podstaw: kroki 1–3 | 15 |
| 60–70 | **Jedyna przerwa** | — |
| 70–85 | **Blok II:** regex, kroki 4–6 | 15 |
| 85–105 | Proste porównanie segmentów przez `groupby` | 20 |
| 105–115 | Małe ćwiczenie z datami | 10 |
| 115–130 | Tabela kandydatów, wnioski i sprawdzenie wyników | 15 |

**Cel bloku I:** rozpoznać tabelę, przeliczyć jednostki, znaleźć kandydatów i odczytać proste wartości z tekstu.

**Cel bloku II:** dokończyć odczyt kart, porównać segmenty, przećwiczyć daty i przygotować krótką rekomendację.

Pracuj nad zadaniami podstawowymi w kolejności. Jeśli skończysz wcześniej, sięgnij do rozszerzeń na końcu. W podsumowaniu ćwiczenia wystarczą dwa konkretne wnioski.

**Poziom:** drugi tydzień; zakładamy znajomość zmiennych, list i uruchamiania komórek. Nowe narzędzia wyjaśniamy na bieżąco.

**Jak pracować:** kod uruchamiaj po kolei (`Shift + Enter`). Wpisuj rozwiązania w komórkach „Twoje rozwiązanie”, a interpretacje w Markdown. Podpowiedzi rozwijaj po samodzielnej próbie. 

Biblioteki: pandas i NumPy. Dane są osadzone jako dokładna kopia CSV z GitHuba prowadzącego; po przygotowaniu środowiska można pracować bez internetu. Przykłady i podglądy tabel są już uruchomione; komórki na rozwiązania pozostają puste.

In [14]:
# Tylko w razie brakujących bibliotek:
# %pip install pandas numpy

In [15]:
import base64
import zlib
from io import StringIO
import re
import numpy as np
import pandas as pd

pd.set_option("display.precision", 2)
pd.set_option("display.max_columns", 12)

## Blok I · 60 minut

## 1. Rozpoznanie danych · 10 minut

Źródło pliku: [kflisikowsky/pandas_exercises — real_estate.csv](https://github.com/kflisikowsky/pandas_exercises/blob/62863036107888c0c7273c40e8d309a43f4da3a9/data/models/real_estate.csv), **414 wierszy, 7 kolumn**. Zbiór dotyczy nieruchomości w New Taipei na Tajwanie. Znaczenie jednostek sprawdzono w [dokumentacji UCI Real Estate Valuation](https://archive.ics.uci.edu/dataset/477/real+estate+valuation+data+set).

| Kolumna źródłowa | Nazwa robocza | Znaczenie |
|---|---|---|
| `No` | `id` | numer rekordu |
| `house age` | `wiek_lat` | wiek budynku w latach, także ułamkowy |
| `distance to the nearest MRT station` | `metro_m` | odległość od najbliższej stacji MRT w metrach |
| `number of convenience stores` | `sklepy` | liczba sklepów w pieszym otoczeniu |
| `latitude`, `longitude` | `lat`, `lon` | współrzędne geograficzne |
| `house price of unit area` | `cena_10k_twd_ping` | cena jednostkowa w **10 000 TWD za ping** |

**To nie jest cena całego lokalu ani cena w PLN/m².** W dokumentacji przyjęto przybliżenie 1 ping = 3,3 m². Do ćwiczenia przeliczymy cenę na TWD/m². W tej kopii nie ma powierzchni, adresów, liczby pokoi ani dat transakcji — nie można ich odtworzyć z pozostałych pól.

`dane_raw` zachowuje oryginał. `df` będzie roboczą kopią z krótszymi nazwami.

In [16]:
DATA_URL = 'https://raw.githubusercontent.com/kflisikowsky/pandas_exercises/62863036107888c0c7273c40e8d309a43f4da3a9/data/models/real_estate.csv'
# Wariant internetowy: dane_raw = pd.read_csv(DATA_URL)

# Osadzona dokładna kopia CSV z tego samego repozytorium:
_csv_blob = 'eJyNXFGyHDlu/FeE7qADVHSQBAmSh7A/HL7AeFfeVcRa2pjR+PzOBAiWVnIU+2dH+x67mgUCiUQCfP/+7fr7tz//+Pzpt799vv765Y/vv339y+dP3799+v73z5++fv7t989/fP/0b//xn5/wm+9fvn29vv75P//1+fdP3/7701++ff3fz1+/fOYH/vj+DSuvf2DN9z//+vn6x7evf/N/+dP/+fsXrMKH/vz65fsnPPW3jx/yJeUa9TX6GOXK6Sr1NUeZ48olv1pNpV7SX/PjB/x2vtolSV9t1n5NX5qk+lKZLV+1vMrHD3JlecnVNGNBbVfzpb3qeqpMLO0v+fihvrO0YX8fP/CXMtOr6ahrXZ/S17qCD1d55Y8f9OqvfJXc2yvJJbZQJTVfmEvDGxUu7JdUvJEWedUu+erxTFlvpLVcNXGb4yr471VGf2kq+Mx6eSxYX1/GVfXVP36Yl+RXvxq+Cd8PO9rSlqZvoOIt8cPBN4K5M2yL/xnY+Yi9ci++1zrwRbbXnLlZfEdqr5Jpc9+s1Lk2Kx2bxfOxtlyKzcJUtWnSdVK9xntVPL8NeypP6qoTpyb4hxtVW162kg7zyqQBcqUF8D9V50tpa1+b0/KUnHBksCRfq/FQYZms2DJM4GtnbutQpWJXeB0+F/9oNFefr5IGLLA8cHlV1TyvhkPH0n7BPbHXOftYJ9B7jYU0a0/2VoNmxc7xuWEv42tbrJWMh8KraSuYT3EEojCIqFxjnUHbjzWvxlYLTgvugneUUeCCfa3s4QM5FXo1fKDgM1xa4IOj9+2EeW0gZzy1mGELo45rYYDci4YXthIhiAdfLb8Ua3FcFS+GjeJFZYZvlRRGGNXsx7WVz82H51YPbQbWS6860kth03VgXVpEwhhYYj6LJ3Hj2AhiYehYBwbjLjPYlxUaAe+NDQh8e2iJkB1ptB9DWwkY3JN5V9dXhUf70tZmfP+Ed4rYe03iUEF84fx6+FavOeKgT+wfXy/JcADn9yoVTrgQo2lEgU7sqdNhEA+lwQmwFmaYl6NgHcX9tc42+opDoGWhqTrCoBL7+vLXkbdvwyZwV0QbgRNGHa/WYYxfIpauhSggYFZ6IUwM59IW8NJHzQEveHw1fxFYr9FUAALRHt+/3x9AiEButldlHGIHCT5QERWJr5lybWttHplHZY/ty7UmwjGluY5VZWhgEeILNoC7ENTKyQmbPXWaC7TeCRX7vUqK9+olQqYmmgAfHHjNUuO8RkkbtapBLMxFswj9NQ1gLJ67Diy35VrA3UwzYbN0SzXgVFh49FhbZo+1XDLsuTgy5X7hBgCeHoC4k4d0GAFHxv1W/gMeBJjvo+dICb1GfEmy48VBAC4LMU6YaLRsn60RCV2R3MS2q9yCmtcMHMbCo57yBiS4LUKRuRNnz1RTFR9FFgug3xbjkVW+1yDIzkuRzDqjTn6KBek8nEyUrfYQ7FHhYL3uYGDSXcGAqIP5AZ6NeAwbwJGRHVu4AvJJi7W0rdC2BDzLi7AXXGRjTEAynaZaMLTCDIqdwdgvZorlYOAivhYQVpiNcAoAKOZy+K4wyma82Q4HPMTAqC2icch1g2dAL6IbPLsicIXPVU/3yHXATqDDMtfU5QbCYMJXc203DAMsEJLAB9bZkpEstgUbIodzDzgyAALgsfYKc3qYt53u8IxMl8GJ4TNCcvJ8CghdoKcmN4PAyVLDdhfS9r5TM/N14ekqzJoJCNht6hsQNIhU7XakcES4dCM7akBGxdIV5m0jfQPOqLkBPa4zzPNPqTFsa/ysGCQo/gFzHcIGFuCWOqw1J1y24fPpp7it4B4LkpSwQY8B1MKyMw5BZacl/LAaOVGCNgwA3KgBXtp6gFcaRk2403HB/ZDDwHcywn45l/Z9sHgUzhL5U+20EDPgRgUxu2w1dQcCNi0Gcx2HVcwJHnk0VwJ+YKlnxgevAlUBELQDEOCX2Gi30ML/wwpASQlDEUN/SMpKQ3WLrnEA5JKwEl+OhcjSiGOSrFVxwO9iB/SmSgc0nyQQPGeaySCgJxnEIXJrwoL1Zr3cz+3cLd/M2DyMmQfyZNm8W0pQmAIsgLsgYAgpw+gGYkLo8csKUSGJvdnkcwfOCz+tmaw38j2SbPDDQRah3C2zU4YZnl17lR6FDkOTISgm8Gb5YS37HOA7/mZDFvOGW06pfXPUEs8tQNdq3B94XixqewIDmCWskIKlg41nI9JYa6TDuVyhg8Zzd9DAnVl+ca3i9IVIqjCee6KOtnMSsdccjH70or3AVvrchVIOc00D7jGcoB/zN02FaGY5hbU4rLVytL6+uzB7V3zzxG/gupPst25n7WOtS6RQjhYIEfy2V7KmXXPVDe2E60ZXhfmmpaFE/9f91Trjq5XmQagipcP59FRIZVqImT7zhTJ4D2EpkkuJDTBElKc5zV1YzAAI55gbgsLuCWS2cqFe4wRU/uXAceDUAVPozTi7gkyN2AMlr+WOvcApbH2SCiGeptVb9ZhbWRyn8wbU9Aa+iScAgCo4iy6jIuDCUpUF9PDF/BlTNqtF7GMT8wD2ArAl9k0uxpe9jknA4ApG5QEluOJMsRLYHd4yKpkSQyQT4rkFhB0sBi9aj5Wtu7Ak86o36bnsNXN1z6wZliWzD7zuKSQSgYtVK3jc2V5HEq88spxM0gDVxSZawn4WBpY4XrzkYroZeA7bZ4o78M0FKTz+BaxEnWHWwoEMWvbkitwBMEFIW/LEjlBgRjHbdyFR6HbTHwwijEgoOOgXQTJwLUyGYCqs4m27lWUPwBXnW+hk6yDydpyB7+6+CTsSJdekqhHUfOwkB+Av5GSUNVhYcXFnDZg18sZIEbsV+ZfljO2488nwAkA58FWCwclSIQBNhk3mvnms4idPGHkTdCvRI9cyDXDpdIjPDbjNUmuZQnei68noCF+vWAGmp8icvji/RaeNHeZS1o4fFxOYM2GZIgtzKBB5B3IOD7JEQ17CxYw35nAkg4Bc0uKl8XRGm61r7hDPCovLQfTpY7wjgZoRrGAAnsKFdcxd0gyJkK+gJepAQppBR3tWQ2y7VgPQ6CaaldjDD5EB11EjnlkS90AuAwLAFNHjucERM+mFaX3ZpYtGCoi9j588EszeJAvuQcoxjGVtQAwmYBMAWhsRxzdMy9D7sZUMCccNiJS9A9Vt28IgtpWNEc/lSvqwA1M3QJUccnOm0GGJH7UWPKxt/t8C+qhNF2MdmcyMaHaQktUXr2CTPF+pp7HJ19yEinKZ+w5hNF2T1ZNqFCF9BLMtrGWrlWy5GrWfp5BwP6fccZZGmmu/hCA6cEOkjo1+veSorxQ/LFbnZu4Mq6wcQN22zy7lMIWCQ66zq/WtKHYFljzWmMtzfvFMQL5zVOpWLNd+iPnmLkHJgyoGTDZGybeMEm9GGcUZWa6Lk0gfcIkNqGXukjSxurYXa4mZjpI9kg18Ox68VT3zHjgxY4P6XiPF7Sy3ct60+Rbr1NCEb0ZiTiF6gHHjuKK/kjd/QBJgXWoPljcUOD9jVMOmLHbQRKL9oiX1fiw7AMnXtne4tisvJuafa75sVVRuluH4FDI02QprUO4yqYEYic9tuPwyKOBRMg9YHVujYA5YDyY7ZoA+RvNiBlQ/WP4enuzRTE7DTERNg4TNDxqZNNbCQZk1CBNaXJQ+VJ+eO+nWeqpnnPdgN2/oCh4Z2k6R0czb1c6NX4aasemuahhkLq+ZMOCkQN/ouYivtCKAPC29kNRqBEZNu5VIwBa3LbMxUxGrSvKD9lOGqUhA6xx6ulhcgPqVl5ZdJtYQ2ODtmXTOOkT8JUKVD79N25eMjsKAn3eyTimEfvOcj71NB4c9yyveUMPJWPLG2wJ4UnACpOZIctU6XrYW3sWlYo3CtlP3xlO2N/zQmIQT8awDKTs9euHZvyrT6lboS/dH5QssufsZ2pcdGmuC6U/Gx/AaIyO/6X41nTt8SB9socmYFC4AQlV0d/Vu6bCZeE1woBRCfB1swMkunOeuxIXJjX7DwzPMASK/yj6zsSohnpmhCF9seGAAgWDcKaHdjTrCbUhMm+n+2bWQdipG3HspbvcTlLkz0EEq1WFlH0ZjcduUQOkEXsCDvXjpBJuhNhu603G4GX2L/+XaTmidZ1nKDDxM6pDU4Xdd9yZCRlRL8zy3Mb1znOG+iTrPr3VhLkvwxBpir5wzADc8Lb2JoV6jFLSePEPF8gZ68Sd7I+DqLZFJ1Sgid9VtHE0cTyejgvoYqMfg4yJxto17kw+mJQBbk00Qq5Pm5kZbIyUH99ZZniZjEVk6HSiOTiSYRmbyKWux+uKqKAio6SypvIaWJlvUQCVGfZ39Z1R7xORV7e02lyultnR43wiZECXZ0Gg4j/FThcGzs2/ANjh10KLqVd2FCzeJrJLZnE7WxDIWpWx/rOfuw2BfVbyN7HJJPevljYuLx/1BhbagI1FibwqMEtE9x8apFi1HoxuS/cnV6MG55Wfv194SV+31TDfmYpz7Xfb2MnZTCBYsxpets1HMJx4V6Wprhxdbqu2VVHY0t9iwKHVU62KVZMyknSUWLs5pBWmHQcqsN63cOIw4cZrGwse8+JHyOA8uuRwFTEudVh8wJQO6sQXA2nJhDSUEL5R5cjyMTFJZTgmRC3lqTBqPEy7rtdRJFFNyz3dnpvwYyMO/36oSqhH4aCG0LLqzBxbocM7CAacXwbSTPtSye+uygyjP1fIoeb5V49iUSynJJyFAXpFle956ULoLalNleQ7FsK1YPdtI5WagcJiswOZ9+EgITPusJYvVDOwfmSRVeS4mZq3F+SaecOO6Vtc3RGofdaH8WG0gwfLg3FaLCZbKdMlge08w4fmyX8FS5JBlixt3UOVBlrBQSzvVpw0kcM1t3OnVJlhuDp2cUww57yzLV3N0oGJCFsPWai4334iQYJLgdrHoKIEYmy1SvD6vqbO40Z00Zb9Y8/EKLmbHhdVupQitbc+HSQlyNkzJtSebqMm2sXDYY+6huxyca5IoF19sCZhCkOboKGmKLA8YMppsr6auPpTGnvUuSzUFojPd+9wXKYf3rE30nDc8BisRvoRmfzDlElQEOEOqa6sI0aDqpRECbOF0iH5eWpNPMiUHEeA+4Fxkp6sYPaysropHe/WuNQDE2vGra12jq8SKpvggk48m9Gcr+ESdNYSAU52dss3L2jYtXG75QjV1kg3Gyno7ONEou2ycRVZ6N52JoP+s1WR/L441NkrEON65RZLSAsYYmgvGal+CBtWPsqP3tkI1Ar6sO3wwAzUuQy0on5a8iRmzpenlhSrHUWdzCGnWDVXOixosu5v3lVVr50xAJiyR9hH32WjuVB6XPsvm/DKZCeRcWmLaolF+CeLCjs+KXlfHaQR8JxvCzznKKKRNhLInDzIOzjPb9ppg05nTNcUpDtGKrVD4G57WN2nZQxyCB3gLrfh0SDsNg3qwO0bgE7TGD32ptkfbbGrQarziHxOWTSAX9zxoi46InbCHu0taj0W8WteiaPLJyUOkrUFAhhnnAOE2t/idotvqTQ2aVxc+4qXGiyqALCllF0GcWilWlxdScZ7bQey03eLYrF56xGgPH7WOGzM7J+x2Y3yP6dRWSkSEGvPvJ5nIqYgaWI8DwxHfhA+PHbTy6hyLWbkfaZ7rraUnZ2+Hg6PVmJg5T4BgQsYumzvtZi37zzmb6FtIvugQj1XFcsn+xjBxTb60rjmFxxbsGpC1CS5ykUkWi29Z+HBP5U1quWYGfWdE1xNFNwkkn+o7rjSELKfC3Cd/XW55bgLzoSPZcBOwH+XCreeETJRZOi+2TYmE/aCBGl3S7ge1HypGXXMInIyjhx1VZAawTVs/jV+YxLqIDlkR6KnuEZixVcjMl3IFznSNdpgTEY/0YZoW8i7rVnr1rqYC7xTPbTaHUcCJzmiuvnR4HfF8sNaVM9ejtGeDt3ev7V8mb20aoczkotqhWK1mhWlsn7pDegkVvxSZPWZ2ODwn9BcWjOytPw/6ucWm+JgDMgwOYueetk9CDKlsB/zJ0jcfJnaKL26MYR5z88Sxh9IibDbcTXWZobAB0ub/g7mtLPHLNSM6zrNiWP3BwxsreBlkSnhf5JNt32YKs+3XTu2o15kUKYnTSC/36m65KGqkvpXpNpfMwJld2zLn0jn0tmi0boYxeWBqY9CJtaq5GfxlU9iyaQunuaT5JsRrn0MKtH6UJB/wRfoGUrH1EGK+7hnzYjpL5WI8lKIdDpDXJ37p8JNRl+lL9a0+ImfXk3ERTqVPu+OzAXrXirPb8KPt16b3WYrCJ8a8tcgfpcjqdnhv/Iej45REkrXBn1pG0+f3sz8WqIlgmjG9jtOKnoZpsr6HbA6KBMUQnT3v4buND0al/SYT1SqO2wu8vjFYY8b6rm9hkj64lM0aOTRrqpW3Qoy0Wy+Pg/Gm1onxZKaKbiXrxl+JI+aIypriz93nMp5rGnux4U/t5MCqYbLeQ8b26wlGAwSJnzXrSSOzKUgppmXxYgEAiEN1cZtD90xNN+GLMeSDJO00zWIDF5Zv2Ss4jHlZIhZTQNipRSoE4bpdeESJSX1WxG912FgSBahOOr1ZsgaNq4ZfBpdifmLlwkOR6bMvbPu8N89vu8DZmNL6lAt03UPxAfHeYDbpd/LcshY927sxLE+NU2f88GUC7E/Ymolv3goBw3hrvsi6RyJ2f9DElN53t2BGknEu6S4vdilEny/ZeLtYeHfGBhOy7Xc3C1q/82db/VRSXCPgj3G30JJjJXi/xCCpsXSmdBN7E/K50m/GwEEovud77FbuwJfVccMaH7oAJ3qZwuN+Wbf40v0Khm13+PUGTWDhHGdZ+x0bUbRZ/5XhIdMLyML24tA9Tqu7dyN5zWALZRJSv+fCNJkdavarCCgcXuPuhPT7GhMrpurXfUwoOTZuuJAldz+QNOOJUhepbLwWcLeLR9nCZSNSmWQntTmiINPAzOFmMSvJrA+otGtUeuiB+1VWU9W9/niKYBNTmI+sWaIM4US8WlG5yvLq1+TWBkAkaNbHmxDNqkFp6Q2/tZtsLft2j9NCdj/MR7cGcvPuE6vEfcpMgaw4nDYDVmqsnWuClGgOVE8UpbzhKDgi63qeim3bggHkaUBFrHQVUsDjYJxPu5nOZPdwOcFV+91YDl6N7dhItu1iHAfRwUvpDtSUzxr2ulOXLutZ4WdwwaAlrUUx2jiNv5gGTcoUhAwEslhu6nkPZZc1kSvqt3dxNrxu0+4G0G554stcphG/RIPiTumYvyZBzvkmrjMxkpRkctD9ztt328xKR7q5tmjmUlCadd+GDjFSVDXSJRUSgmMpIL9VNk3ez818W7+ZwUuKuVrrI5ndd6UZR9Hsyo/VpEI9hj0juChZxy9CXJO8+kVGRW3i5PGar98Jl86i/FzDO/b3/I4GVT0R89o1uRHTHJ53310N2OGdDb8sJt1usc5DRxd+QLv1emrdeY+PU8LtPNDoh9F1qb3cApwmhKIc17oaqxO/x01a4ZdCO91tyj6O6EqCb881py4cJLGJxsd5KfUH2/Vr2Ip3iHkLLFhR5PeqEoNYMtJZvjRJx4q+er6450E0OAmROUpcdTdW9h8+EJJD/xMFRnc4FCHs9LU92Sllp3fmKlerxHUroDl1bx5ciId7KGKMua72cOjHRt0Ow2Dr1q2V3mCQcLShxOhf0CSaXDKOY5KOkX67hmGRgH2t3cPyERY2LO/Mxbo/hKgCd577Xn+bdyOkSVQiM60LKWwH6K0c9rZLEYCNU/WZvWnCK06DZfUiDtS717nZ9XNuYhZrPcMbp11P3WAS/st3X3lo2pQRMiM7tls463PfnBrdBgltbY3a4gFTfWVIyZnX9PomBIE7lSiwMsBUlw4fb716VTGNnQDUWexYpltGCJdsRjit0Wb3RHwY6/FPLDg+MEWfw5jicE02l3y8G+N/uiNZN2KcRr9tjISXdylQP/5ZjrJugsvaxGPl5lpQTdXvTmS2IGu4zrj/dMFUXZMANbV31AfnkrQfv5oPn/VWcu+bksyY3ZfyJtu0EYPMMF2pcw8Qe2Qm3++w4Y0DfyhuYMtwzY1GxAur7cFk9iWtdqw5eTfzOI5buTgbiXkm6okL7WrdsNzCbslmBZumrza8crG8c6PHvL1mH0t+vtxl7av/A6aHi9Q='
dane_raw = pd.read_csv(StringIO(zlib.decompress(base64.b64decode(_csv_blob)).decode("utf-8-sig")))
df = dane_raw.rename(columns={
    "No": "id", "house age": "wiek_lat",
    "distance to the nearest MRT station": "metro_m",
    "number of convenience stores": "sklepy",
    "latitude": "lat", "longitude": "lon",
    "house price of unit area": "cena_10k_twd_ping"
}).copy()
print(f"Załadowano {len(df)} nieruchomości; {df.shape[1]} kolumn.")
df.head(6)

Załadowano 414 nieruchomości; 7 kolumn.


,id,wiek_lat,metro_m,sklepy,lat,lon,cena_10k_twd_ping
0,1,32.0,84.88,10,24.98,121.54,37.9
1,2,19.5,306.59,9,24.98,121.54,42.2
2,3,13.3,561.98,5,24.99,121.54,47.3
3,4,13.3,561.98,5,24.99,121.54,54.8
4,5,5.0,390.57,5,24.98,121.54,43.1
5,6,7.1,2175.03,3,24.96,121.51,32.1


In [65]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 414 non-null    int64  
 1   wiek_lat           414 non-null    float64
 2   metro_m            414 non-null    float64
 3   sklepy             414 non-null    int64  
 4   lat                414 non-null    float64
 5   lon                414 non-null    float64
 6   cena_10k_twd_ping  414 non-null    float64
dtypes: float64(5), int64(2)
memory usage: 22.8 KB


### Zadanie 1 · Kontrola na wejściu

Wyświetl wymiary tabeli, typy kolumn, liczbę braków i statystyki opisowe. Sprawdź unikatowość `id`. Które zmienne są liczbowe, ale nie powinny być interpretowane jako zwykłe miary ilościowe? Czy liczba braków równa 0 oznacza, że wszystkie dane są na pewno poprawne?

<details><summary>Mała podpowiedź</summary>

`df.isna().sum()`, `df.describe()`, `df["id"].is_unique`.

</details>

In [66]:
# Twoje rozwiązanie
df.isna().sum()

id                   0
wiek_lat             0
metro_m              0
sklepy               0
lat                  0
lon                  0
cena_10k_twd_ping    0
dtype: int64

In [67]:
df.describe().round(2)

,id,wiek_lat,metro_m,sklepy,lat,lon,cena_10k_twd_ping
count,414.00,414.00,414.00,414.00,414.00,414.00,414.00
mean,207.50,17.71,1083.89,4.09,24.97,121.53,37.98
std,119.66,11.39,1262.11,2.95,0.01,0.02,13.61
min,1.00,0.00,23.38,0.00,24.93,121.47,7.60
25%,104.25,9.02,289.32,1.00,24.96,121.53,27.70
50%,207.50,16.10,492.23,4.00,24.97,121.54,38.45
75%,310.75,28.15,1454.28,6.00,24.98,121.54,46.60
max,414.00,43.80,6488.02,10.00,25.01,121.57,117.50


In [68]:
df["id"].nunique()

414

**Twoja odpowiedź / interpretacja:**

_Uzupełnij tę komórkę Markdown (dwuklik)._

## 2. Operatory i jednostki · 15 minut

`=` przypisuje, `==` porównuje. Potęga to `**`, dzielenie `/`, dzielenie całkowite `//`, reszta `%`. Python używa kropki dziesiętnej. Nazwy takie jak `sum`, `list` czy `str` zachowaj dla funkcji wbudowanych.

Kolumna pandas (`Series`) pozwala wykonywać działania na wszystkich rekordach naraz. Przykład przelicza odległość z metrów na kilometry i wskazuje nieruchomości blisko metra.

In [17]:
df["metro_km"] = df["metro_m"] / 1000
df["blisko_metra"] = df["metro_m"] <= 500
print("Najkrótsza odległość [m]:", df["metro_m"].min())
print("Liczba blisko metra:", df["blisko_metra"].sum())
print("Odsetek blisko metra [%]:", 100 * df["blisko_metra"].mean())
df[["id", "metro_m", "metro_km", "blisko_metra"]].head()

Najkrótsza odległość [m]: 23.38284
Liczba blisko metra: 211
Odsetek blisko metra [%]: 50.96618357487923


,id,metro_m,metro_km,blisko_metra
0,1,84.88,0.08,True
1,2,306.59,0.31,True
2,3,561.98,0.56,False
3,4,561.98,0.56,False
4,5,390.57,0.39,True


### Zadanie 2 · Cena w porównywalnych jednostkach

Przyjmij `M2_NA_PING = 3.3`. Dodaj `cena_twd_m2`: pomnóż cenę źródłową przez 10000 i podziel przez liczbę m² w pingu. Pokaż źródłową i przeliczoną cenę dla 5 rekordów. Nie przeliczaj waluty na PLN. Dlaczego nie wolno z tej tabeli obliczyć ceny całego lokalu?

In [18]:
# Twoje rozwiązanie
df["cena_twd_m2"] = df["cena_10k_twd_ping"] * 10000 / 3.3058
df["cena_twd_m2"] = df["cena_twd_m2"].round(2)
df[["id", "cena_twd_m2", "cena_10k_twd_ping"]].head(5)


,id,cena_twd_m2,cena_10k_twd_ping
0,1,114646.98,37.9
1,2,127654.43,42.2
2,3,143081.86,47.3
3,4,165769.25,54.8
4,5,130376.91,43.1


### Zadanie 3 · Dostępność usług

Dodaj `duzo_sklepow` oznaczającą co najmniej 5 sklepów. Policz liczbę i procent takich nieruchomości. Następnie sprawdź, czy wszystkie odległości są nieujemne oraz czy przynajmniej jeden budynek ma wiek 0.

<details><summary>Mała podpowiedź</summary>

Dla maski `True/False`: `.sum()` zlicza, `.mean()` oblicza udział, `.all()` pyta o wszystkie, `.any()` — o choć jeden rekord.

</details>

In [73]:
# Twoje rozwiązanie
df["duzo_sklepow"] = df["sklepy"] >= 5
print("Liczba duzo_sklepow:", df["duzo_sklepow"].sum())
print("Odsetek duzo_sklepow [%]:", 100 * df["duzo_sklepow"].mean())

Liczba duzo_sklepow: 200
Odsetek duzo_sklepow [%]: 48.309178743961354


### Dlaczego nie lista?

Mnożenie listy przez 2 powtarza elementy. Tablica NumPy i kolumna pandas mnożą wartości. Prawy koniec `range()` jest pomijany.

In [41]:
odleglosci = [100, 300, 500]
print(odleglosci * 2)
print(np.array(odleglosci) * 2)
print(list(range(1, 6)))

[100, 300, 500, 100, 300, 500]
[ 200  600 1000]
[1, 2, 3, 4, 5]


## 3. Filtry: nieruchomości dla klienta · 20 minut

`and`, `or`, `not` łączą pojedyncze warunki. Dla całych kolumn stosuj **`&`, `|`, `~`**, otaczając każde porównanie nawiasami. `.loc[maska, kolumny]` wybiera wiersze i kolumny. `between(a, b)` uwzględnia obie granice, a `isin([...])` bada przynależność do listy wartości.

Przykład: budynki mające do 10 lat, z metrem do 500 m, posortowane od najbliższego.

In [19]:
maska_przyklad = (df["wiek_lat"] <= 10) & (df["metro_m"] <= 500)
(df.loc[maska_przyklad, ["id", "wiek_lat", "metro_m", "sklepy", "cena_10k_twd_ping"]]
   .sort_values("metro_m")
   .head(5))

,id,wiek_lat,metro_m,sklepy,cena_10k_twd_ping
19,20,1.5,23.38,7,47.7
275,276,1.5,23.38,7,49.7
160,161,3.5,49.66,8,57.8
377,378,3.9,49.66,8,56.8
291,292,3.4,56.47,7,54.4


### Zadanie 4 · Wymagania klienta

Zapisz `maska_klienta`: wiek do 20 lat, metro do 800 m i co najmniej 3 sklepy. Utwórz `kandydaci = df.loc[maska_klienta].copy()`. Policz kandydatów. Pokaż 5 z najniższą ceną TWD/m²; przy tej samej cenie pierwszeństwo ma krótsza odległość do metra.

<details><summary>Mała podpowiedź</summary>

`.sort_values(["cena_twd_m2", "metro_m"], ascending=[True, True]).head(5)`.

</details>

In [20]:
# Twoje rozwiązanie
maska_klienta = (df["wiek_lat"] <= 20) & (df["metro_m"] <= 800) & (df["sklepy"] >= 3)
kandydaci = df.loc[maska_klienta].copy()
print("Liczba kandydatów:", len(kandydaci))
kandydaci[["id", "wiek_lat", "metro_m", "sklepy", "cena_10k_twd_ping"]].sort_values(["cena_10k_twd_ping","metro_m"]).head(5)



Liczba kandydatów: 138


,id,wiek_lat,metro_m,sklepy,cena_10k_twd_ping
113,114,14.8,393.26,6,7.6
292,293,17.5,395.67,5,24.5
271,272,17.7,451.64,8,26.5
120,121,13.3,492.23,5,31.3
199,200,18.2,451.64,8,31.6


### Zadanie 5 · Kompromisy

Znajdź nieruchomości położone do 300 m od metra **lub** mające co najmniej 8 sklepów. Wyklucz budynki starsze niż 30 lat. Zapisz maskę i pokaż liczbę rekordów. Jak zmieni się wynik po zamianie „lub” na „i”? Sprawdź obie wersje.

In [21]:
# Twoje rozwiązanie
maska_klienta2 = ((df["metro_m"] <= 300) | (df["sklepy"] >= 8)) & (df["wiek_lat"] <= 30)
kandydaci2 = df.loc[maska_klienta2].copy()
print("Liczba kandydatów2:", len(kandydaci2))

maska_klienta3 = (df["metro_m"] <= 300) & (df["sklepy"] >= 8) & (df["wiek_lat"] <= 30)
kandydaci3 = df.loc[maska_klienta3].copy()
print("Liczba kandydatów3:", len(kandydaci3))

Liczba kandydatów2: 106
Liczba kandydatów3: 17


### Zadanie 6 · Odwróć filtr

Wybierz budynki w wieku od 10 do 20 lat włącznie, z liczbą sklepów należącą do `[0, 1, 2]`. Następnie przez `~maska_klienta` policz rekordy niespełniające wymagań klienta. Sprawdź, czy kandydaci i pozostałe rekordy razem dają 414.

<details><summary>Mała podpowiedź</summary>

`df["wiek_lat"].between(10, 20)` i `df["sklepy"].isin([0, 1, 2])`.

</details>

In [24]:
# Twoje rozwiązanie
maska_klienta4 = (df["sklepy"].isin([0,1,2])) & (df["wiek_lat"] >= 10) & (df["wiek_lat"] <= 20)
~maska_klienta4.count()

np.int64(-415)

**Twoja odpowiedź / interpretacja:**

_Uzupełnij tę komórkę Markdown (dwuklik)._

**Zabawa w parach — w ramach tego bloku:** jedna osoba opisuje „wymagającego klienta”, druga zapisuje filtr. Warunki muszą zawierać co najmniej jedno „lub”. Porównajcie, czy nawiasy odpowiadają intencji klienta.

## 4. Regex: małe pytanie, krótki kod · 2 × 15 minut

Pracujemy na **tych samych nieruchomościach z `df`**. Ich wartości zapiszemy w krótkich kartach tekstowych, jak w eksporcie z systemu. Nie dodajemy adresów ani cech, których nie ma w CSV. Kod `RE-0001` jest technicznym oznaczeniem rekordu.

Każde zadanie to jedno małe pytanie. Zwykle wystarczy **jedna lub dwie linie**. Najpierw wydobądź informację, potem ją wyświetl. Przed przerwą zrób część A, po przerwie część B.

In [7]:
eksport = df[["id", "wiek_lat", "metro_m", "sklepy", "cena_10k_twd_ping"]].copy()
eksport["kod"] = "RE-" + eksport["id"].astype(str).str.zfill(4)
eksport["karta"] = (
    eksport["kod"]
    + " | wiek=" + eksport["wiek_lat"].astype(str)
    + " | metro=" + eksport["metro_m"].astype(str)
    + " | sklepy=" + eksport["sklepy"].astype(str)
    + " | cena=" + eksport["cena_10k_twd_ping"].astype(str)
)
eksport[["id", "karta"]].head(6)

,id,karta
0,1,RE-0001 | wiek=32.0 | metro=84.87882 | sklepy=...
1,2,RE-0002 | wiek=19.5 | metro=306.5947 | sklepy=...
2,3,RE-0003 | wiek=13.3 | metro=561.9845 | sklepy=...
3,4,RE-0004 | wiek=13.3 | metro=561.9845 | sklepy=...
4,5,RE-0005 | wiek=5.0 | metro=390.5684 | sklepy=5...
5,6,RE-0006 | wiek=7.1 | metro=2175.03 | sklepy=3 ...


### Regex pod ręką — tabela podstaw

Nie ucz się całej tabeli na pamięć. W zadaniach podstawowych wystarczą przede wszystkim **cyfry, nawiasy i `+`**. Pozostałe zapisy potraktuj jako ściągę do własnych prób.

| Zapis | Co oznacza? | Mały wzorzec | Co znajdzie? |
|---|---|---|---|
| `metro=` | zwykły tekst | `metro=` | dokładnie fragment `metro=` |
| `[0-9]` | jedna cyfra od 0 do 9 | `sklepy=[0-9]` | `sklepy=5`; z `sklepy=10` dopasuje tylko `sklepy=1` |
| `\d` | cyfra; w Pythonie także niektóre cyfry Unicode | `\d+` | np. `414`; w naszych kartach wystarczy `[0-9]+` |
| `+` | jeden lub więcej poprzednich elementów | `[0-9]+` | `5`, `10`, `414` |
| `*` | zero lub więcej poprzednich elementów | `\s*` | brak spacji albo dowolną liczbę białych znaków |
| `?` | poprzedni element jest opcjonalny | `RE-?` | `RE` lub `RE-` |
| `{4}` | dokładnie cztery powtórzenia | `[0-9]{4}` | np. `0001` |
| `{1,3}` | od jednego do trzech powtórzeń | `[0-9]{1,3}` | ciąg od 1 do 3 cyfr; sam wzorzec nie sprawdza całego pola |
| `[0-9.]` | jedna cyfra albo dosłowna kropka | `[0-9.]+` | np. `84.87882` w naszych kartach |
| `.` | dowolny znak poza nową linią przy ustawieniach domyślnych | `RE.0001` | np. `RE-0001`, ale także `REX0001` |
| `\.` | dosłowna kropka | `[0-9]+\.[0-9]+` | np. `32.0` |
| `\s` | biały znak, np. spacja lub tabulator | `metro=\s*[0-9]+` | np. `metro=84` i `metro= 84` |
| `^` | początek napisu | `^RE-` | `RE-` wyłącznie na początku karty |
| `$` | koniec napisu; w Pythonie także pozycja przed końcowym znakiem nowej linii | `cena=([0-9.]+)$` | cenę w ostatnim polu karty |
| `( ... )` | grupa przechwytująca — fragment do wydobycia | `sklepy=([0-9]+)` | dopasowanie `sklepy=10`, **grupa 1: `10`** |
| `A\|B` | alternatywa: A lub B | `wiek\|metro` | słowo `wiek` albo `metro` |
| `[^0-9]` | znak inny niż cyfra 0–9 | `[^0-9]+` | fragmenty bez cyfr |

**Uwaga o kropce:** `[0-9.]+` jest prostym odczytem dla znanego formatu. Może też dopasować błędny zapis `12..3`; nie jest pełnym sprawdzaniem poprawności liczby. W razie potrzeby weryfikuj wynik przez `pd.to_numeric()`.

**W pandas:** `.str.contains()` odpowiada „czy występuje?”, a `.str.extract()` — „co jest w nawiasie?”. Do wydobycia jednej informacji używaj jednej grupy przechwytującej i `expand=False`.

### Jak sprawdzić wzorzec na regex101.com?

Otwórz **[regex101.com — tester wyrażeń regularnych](https://regex101.com/)**. Strona podświetla dopasowania, pokazuje przechwycone grupy i objaśnia elementy wzorca. Możesz używać jej obok notebooka podczas ćwiczeń.

1. Wybierz **Flavor: Python**, aby testować składnię zgodną z Pythonem.
2. Do pola tekstu testowego (**Test String / Input**) wklej **jedną kartę** z notebooka. Na początek możesz użyć tej:

   ```text
   RE-0001 | wiek=32.0 | metro=84.87882 | sklepy=10 | cena=37.9
   ```

3. Do pola **Regular Expression** wpisz sam wzorzec:

   ```text
   sklepy=([0-9]+)
   ```

   **Na stronie nie dopisuj `r`, cudzysłowów ani ukośników otaczających wzorzec.** W Pythonie `r"..."` jest sposobem zapisu napisu w kodzie, a nie częścią samego regexu.
4. Sprawdź podświetlenie: całe dopasowanie to `sklepy=10`. W **Match Information** odszukaj **Group 1** — powinna zawierać `10`.
5. Zamień wzorzec na `metro=([0-9.]+)`. Teraz grupa 1 powinna zawierać `84.87882`. Zmień w tekście odległość na `250.5` i zobacz, czy odczyt nadal działa.
6. Wróć do notebooka i przenieś **ten sam wzorzec** do `str.extract()`:

   ```python
   eksport["karta"].str.extract(r"metro=([0-9.]+)", expand=False).head()
   ```

**Gdy nic nie pasuje:** sprawdź, czy etykieta w tekście jest identyczna, np. `metro=` zamiast `Metro:`, oraz czy nie pojawiła się dodatkowa spacja. Zmieniaj jeden element naraz. W pandas sprawdzasz wzorzec na każdym wierszu osobno; na stronie zacznij od jednej karty, żeby nie komplikować ustawień.

**Mała próba:** wpisz `wiek=([0-9]+)`, a następnie `wiek=([0-9.]+)`. Porównaj zawartość grupy 1 dla `wiek=32.0`. Który wzorzec odczytał cały wiek?


### Przykład na start: wyciągnij liczbę sklepów

Wzorzec `sklepy=([0-9]+)` czytamy: „znajdź `sklepy=`, a potem zabierz cyfry do nawiasu”. `[0-9]` oznacza cyfrę, `+` — jedną lub więcej. Nawias wskazuje, **co chcemy dostać w wyniku**.

Nie trzeba odczytywać całej karty. Szukamy jednego krótkiego fragmentu.

In [8]:
eksport["karta"].head(3)

0    RE-0001 | wiek=32.0 | metro=84.87882 | sklepy=...
1    RE-0002 | wiek=19.5 | metro=306.5947 | sklepy=...
2    RE-0003 | wiek=13.3 | metro=561.9845 | sklepy=...
Name: karta, dtype: object

In [9]:
eksport["karta"].str.extract(r"sklepy=([0-9]+)", expand=False).head()

0    10
1     9
2     5
3     5
4     5
Name: karta, dtype: object

## Część A · 15 minut

W kolejnych zadaniach korzystaj z `eksport["karta"]`. `.head()` ogranicza wyświetlanie do 5 pierwszych wyników.

### 7A · Jaki to rekord?

Wyciągnij z kart sam numer po `RE-`. Wyświetl 5 pierwszych numerów.

<details><summary>Mała podpowiedź</summary>

Wzorzec: `r"RE-([0-9]+)"`. Użyj `.str.extract(..., expand=False)`.

</details>

In [ ]:
# Twoje rozwiązanie


### 7B · Pokaż pełne oznaczenie

Teraz wyciągnij pełny kod, np. `RE-0001`, zamiast samych cyfr. Wystarczy zmienić położenie nawiasów we wzorcu.

<details><summary>Mała podpowiedź</summary>

To, co ma trafić do wyniku, umieść w nawiasach: `r"(RE-[0-9]+)"`.

</details>

In [ ]:
# Twoje rozwiązanie


### 7C · Ile jest sklepów?

Odczytaj liczbę sklepów z **pierwszej karty**. Wykorzystaj pokazany przykład i na końcu wybierz `.iloc[0]`. Zmień na `.iloc[4]`, żeby odczytać piątą kartę.

In [ ]:
# Twoje rozwiązanie


### 7D · Znajdź karty bez sklepów

Pokaż 5 kart zawierających tekst `sklepy=0`. Tu nie musisz niczego wydobywać — wystarczy małe wyszukiwanie.

<details><summary>Mała podpowiedź</summary>

`maska = eksport["karta"].str.contains("sklepy=0", regex=False)`

Następnie: `eksport.loc[maska, "karta"].head()`.

</details>

In [ ]:
# Twoje rozwiązanie


## Przerwa · 10 minut

**Koniec bloku I — 60 minut pracy.** Po przerwie zostajemy przy tych samych kartach.

## Blok II · 60 minut

Regex: część B (15 min), `groupby` (20 min), daty (10 min), podsumowanie analizy (15 min).

## Część B · 15 minut

### Jeszcze jeden mały wzorzec: liczba z kropką

`[0-9.]+` odczytuje cyfry i kropki w znanym formacie naszych kart. Kropka w nawiasach kwadratowych oznacza zwykłą kropkę. To prosta reguła do tego eksportu, nie uniwersalny wzorzec dla wszystkich liczb.

Przykład odczytuje odległość i zamienia wynik na liczbę. Dzięki temu można później porównywać wartości.

In [10]:
metro_tekst = eksport["karta"].str.extract(r"metro=([0-9.]+)", expand=False)
eksport["metro_odczytane"] = pd.to_numeric(metro_tekst)
eksport["metro_odczytane"].head()

0     84.88
1    306.59
2    561.98
3    561.98
4    390.57
Name: metro_odczytane, dtype: float64

### 8A · Ile lat ma budynek?

Wyciągnij wiek budynku z 5 pierwszych kart. Weź wzorzec z przykładu i zmień tylko etykietę `metro=` na `wiek=`.

In [ ]:
# Twoje rozwiązanie


### 8B · Jaka cena?

Wyciągnij cenę z kart i wyświetl 5 pierwszych wartości. Nie zmieniaj jednostki — to nadal cena źródłowa w 10 000 TWD/ping.

<details><summary>Mała podpowiedź</summary>

Zacznij od `r"cena=..."`; w nawiasie zostaw `[0-9.]+`.

</details>

In [ ]:
# Twoje rozwiązanie


### 8C · Jaka jest najwyższa cena?

Zapisz odczytane ceny jako `ceny_tekst`. Zamień je na liczby przez `pd.to_numeric(ceny_tekst)` i wyświetl największą przez `.max()`.

<details><summary>Mała podpowiedź</summary>

Dwie krótkie operacje: ekstrakcja, a potem konwersja i `.max()`.

</details>

In [ ]:
# Twoje rozwiązanie


### 9 · Które rekordy mają metro do 500 m?

Korzystając z gotowej kolumny `metro_odczytane`, pokaż 5 kart z odległością do 500 m. Regex już wykonał odczyt — teraz wystarczy zwykły filtr.

<details><summary>Mała podpowiedź</summary>

`eksport.loc[eksport["metro_odczytane"] <= 500, "karta"].head()`.

</details>

In [ ]:
# Twoje rozwiązanie


### 10 · Własne małe zapytanie

Wybierz **jedno** pytanie i odpowiedz krótkim kodem:

- Jaki jest wiek budynku z karty o `id == 10`?
- Jakie są trzy najniższe ceny odczytane z kart?
- Ile kart ma co najmniej 5 sklepów?

Wystarczy odczytać jedną wartość lub jedną kolumnę, zamienić na liczbę, jeśli potrzeba, i wykonać proste działanie.

<details><summary>Mała podpowiedź</summary>

Do odczytu jednej karty: `eksport.loc[eksport["id"] == 10, "karta"]`.

Do trzech najniższych liczb: `.nsmallest(3)`. Do zliczenia maski: `.sum()`.

</details>

In [ ]:
# Twoje rozwiązanie


### Ściąga: wystarczą trzy wzorce

| Co wyciągasz? | Wzorzec |
|---|---|
| Numer po `RE-` | `r"RE-([0-9]+)"` |
| Liczbę sklepów | `r"sklepy=([0-9]+)"` |
| Wiek, odległość lub cenę | `r"wiek=([0-9.]+)"`, analogicznie `metro=` lub `cena=` |

**Schemat:** `kolumna.str.extract(wzorzec, expand=False)` → w razie potrzeby `pd.to_numeric(...)` → małe pytanie o wynik.

Bez dopasowania dostaniesz brak wartości. Nie musisz tu tworzyć skomplikowanych grup, łączyć kilku pól ani budować nowej analizy od początku.

## 5. `groupby`: segmenty rynku · 20 minut

Stworzymy trzy segmenty dostępności metra: **do 500 m**, **powyżej 500 do 1500 m** oraz **powyżej 1500 m**. Granice są założeniem analitycznym ćwiczenia, nie kategoriami dostarczonymi przez autora zbioru.

`pd.cut()` tworzy przedziały. `groupby()` dzieli dane na grupy, a `agg()` liczy podsumowania. `size` liczy wiersze, natomiast `count` pomija braki w liczonej kolumnie. W przykładach podajemy `observed=True`, aby pokazywać tylko zaobserwowane kombinacje kategorii.

In [11]:
df["strefa_metra"] = pd.cut(
    df["metro_m"], bins=[-np.inf, 500, 1500, np.inf],
    labels=["do 500 m", "500–1500 m", "powyżej 1500 m"], right=True
)
przyklad_grup = (
    df.groupby("strefa_metra", observed=True, as_index=False)
      .agg(liczba=("id", "size"), mediana_ceny_zrodlowej=("cena_10k_twd_ping", "median"))
)
przyklad_grup

,strefa_metra,liczba,mediana_ceny_zrodlowej
0,do 500 m,211,44.8
1,500–1500 m,104,36.1
2,powyżej 1500 m,99,23.5


### Zadanie 11 · Bliskość metra a cena

Zbuduj `raport_stref`: dla każdej `strefa_metra` oblicz liczebność, średnią i medianę `cena_twd_m2`, medianę wieku oraz średnią liczbę sklepów. Posortuj wynik po medianie ceny malejąco. Czy najdroższa strefa ma też najwięcej obserwacji? Podaj liczby.

<details><summary>Mała podpowiedź</summary>

Nazwane agregacje: `.agg(liczba=("id", "size"), mediana_ceny=("cena_twd_m2", "median"), ...)`.

</details>

In [ ]:
# Twoje rozwiązanie


### Małe ćwiczenie z datą: termin ponownego sprawdzenia

Zostańmy przy trzech pierwszych nieruchomościach. **Wyłącznie do ćwiczenia** przyjmujemy, że analityk sprawdził je 1, 12 i 15 października 2026 r. To daty pracy analityka, nie daty transakcji z CSV.

Chcemy ustalić, ile dni minęło od sprawdzenia do 19.10.2026. W tym małym zadaniu nie potrzebujemy regexu: daty są już w osobnej kolumnie.

In [12]:
monitoring = df[["id", "wiek_lat", "metro_m"]].head(3).copy()
monitoring["sprawdzono"] = pd.to_datetime(
    ["01.10.2026", "12.10.2026", "15.10.2026"], format="%d.%m.%Y"
)
dzien_analizy = pd.Timestamp("2026-10-19")
monitoring["ile_dni"] = (dzien_analizy - monitoring["sprawdzono"]).dt.days
monitoring

,id,wiek_lat,metro_m,sprawdzono,ile_dni
0,1,32.0,84.88,2026-10-01,18
1,2,19.5,306.59,2026-10-12,7
2,3,13.3,561.98,2026-10-15,4


**Zadanie 15 · Krótkie ćwiczenie.**

1. Wybierz rekordy niesprawdzane od **co najmniej 7 dni**.
2. Dodaj kolumnę `sprawdz_ponownie`: data poprzedniego sprawdzenia + 7 dni. Użyj `pd.Timedelta(days=7)`.

Odpowiedz: które rekordy mają już termin ponownego sprawdzenia do 19.10.2026 włącznie?

<details><summary>Mała podpowiedź</summary>

Filtr: `monitoring.loc[monitoring["ile_dni"] >= 7]`. Dodawanie: `monitoring["sprawdzono"] + pd.Timedelta(days=7)`.

</details>

In [ ]:
# Twoje rozwiązanie


## 7. Podsumowanie analizy · 15 minut

Wróć do początkowych wymagań: wiek ≤20 lat, metro ≤800 m, co najmniej 3 sklepy. Korzystaj z **tych samych rekordów źródłowych** i wyników zadań, nie wymyślaj brakujących cech.

**Uwaga techniczna:** `kandydaci` to kopia z zadania 4, więc nie dostanie automatycznie kolumn dodanych później do `df`. Gdy potrzebujesz nowych kolumn, ponownie wybierz rekordy z `df` przez `maska_klienta`.

### Zadanie 16 · Krótka lista kandydatów

Zbuduj `rekomendacja`: pięć nieruchomości spełniających warunki klienta o najniższej cenie TWD/m², z rozstrzyganiem remisów po odległości od metra. Pokaż id, wiek, metro, sklepy i cenę jednostkową. Nadaj kolumnom czytelne etykiety przez `rename()` i zaokrąglij widok do 1 miejsca. Nie zaokrąglaj kolumn źródłowych przed wyborem.

<details><summary>Mała podpowiedź</summary>

Wykorzystaj `.loc[...]`, `.sort_values([...])`, `.head(5)`, `.rename(columns={...})`, `.round(1)`.

</details>

In [ ]:
# Twoje rozwiązanie


**Zadanie 17 — wnioski dla klienta:** napisz 2–3 zdania: ile rekordów spełnia wymagania, jaki kandydat ma najniższą cenę jednostkową, jaki kompromis widać w porównaniu segmentów i czego brakuje do rzeczywistej decyzji zakupowej. W co najmniej dwóch zdaniach podaj konkretne liczby i jednostki. Pamiętaj, że nie znamy metrażu ani ceny całego lokalu.

**Twoja odpowiedź / interpretacja:**

_Uzupełnij tę komórkę Markdown (dwuklik)._

### Sprawdzenie wyników · w ramach ostatnich 15 minut

Odkomentuj kontrolę po wykonaniu zadań podstawowych. Drobne różnice zmiennoprzecinkowe porównujemy przez `np.isclose` i `np.allclose`.

In [ ]:
# Odkomentuj po wykonaniu odpowiednich zadań:
# assert dane_raw.shape == (414, 7)
# assert df["id"].is_unique
# assert np.allclose(df["cena_twd_m2"], df["cena_10k_twd_ping"] * 10000 / 3.3)
# assert len(kandydaci) == int(maska_klienta.sum())
# assert np.allclose(eksport["metro_odczytane"], eksport["metro_m"])
# assert raport_stref["liczba"].sum() == len(df)
# print("Kontrole zakończone poprawnie.")


## Rozszerzenia · tylko po wykonaniu części podstawowej

Poniższe zadania są **poza planem 2 × 60 minut**. Numery zadań 12–14 pozostawiono dla łatwego odwoływania się do materiału. Do podstawowej rekomendacji wystarczy tabela `raport_stref` z zadania 11.

### Zadanie 12 · Porównaj podobne z podobnym

Dodaj `grupa_wieku` przez `pd.cut`: do 10 lat włącznie, powyżej 10 do 20, powyżej 20. Grupuj jednocześnie po `strefa_metra` i `grupa_wieku`. Oblicz liczebność i medianę ceny TWD/m². Zapisz `raport_segmentow`. Wyświetl tylko segmenty mające co najmniej 10 rekordów. Dlaczego porównanie z kontrolą wieku może dawać inny obraz niż samo porównanie stref?

<details><summary>Mała podpowiedź</summary>

Użyj `bins=[-np.inf, 10, 20, np.inf]`, `labels=["do 10 lat", "10–20 lat", "powyżej 20 lat"]`, a potem `groupby(["strefa_metra", "grupa_wieku"], observed=True)`.

</details>

In [ ]:
# Twoje rozwiązanie


**Twoja odpowiedź / interpretacja:**

_Uzupełnij tę komórkę Markdown (dwuklik)._

### Tabela przestawna bez Excela

`pivot_table()` układa wynik w macierz: jedna cecha w wierszach, druga w kolumnach, a w środku wybrana agregacja. Brak obserwacji w kombinacji to brak danych, a nie cena 0.

### Zadanie 13 · Mapa segmentów w tabeli

Utwórz `pivot_table`: wiersze `grupa_wieku`, kolumny `strefa_metra`, wartości `cena_twd_m2`, agregacja `median`, `observed=True`. Obok przygotuj liczebności przez `pd.crosstab()`. Którą komórkę porównania uznajesz za najmniej pewną i dlaczego?

<details><summary>Mała podpowiedź</summary>

`df.pivot_table(index=..., columns=..., values=..., aggfunc="median", observed=True)`.

</details>

In [ ]:
# Twoje rozwiązanie


### Rozszerzenie · Własne funkcje i standaryzacja

Poza podstawowymi 120 minutami pracy. Krótka funkcja i pętla pozwalają powtórzyć te same działania dla kilku progów. Następnie możesz zbudować własną funkcję standaryzującą.

In [13]:
def metry_na_kilometry(odleglosc_m):
    return odleglosc_m / 1000

for prog_m in [300, 500, 800]:
    liczba = (df["metro_m"] <= prog_m).sum()
    print(f"Do {metry_na_kilometry(prog_m):.1f} km od metra: {liczba} rekordów")

Do 0.3 km od metra: 116 rekordów
Do 0.5 km od metra: 211 rekordów
Do 0.8 km od metra: 261 rekordów


### Standaryzacja cen

$$z_i=\frac{x_i-\bar{x}}{s},\qquad s=\sqrt{\frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}}.$$

Używamy odchylenia standardowego z próby (`ddof=1`). Wynik jest bez jednostki. Wartość ujemna oznacza cenę poniżej średniej, a nie ujemną cenę. Standaryzacja **nie zmienia danych w rozkład normalny**.

### Zadanie 14 · Własna funkcja

Napisz `standaryzuj(x)` z użyciem `.mean()` i `.std(ddof=1)`. Dla zerowego odchylenia zgłoś `ValueError`. Zakładamy numeryczną kolumnę z minimum dwiema obserwacjami i bez braków. Dodaj `cena_z` do `df`. Sprawdź średnią i odchylenie wyniku oraz przetestuj funkcję na `pd.Series([10, 20, 30])`.

<details><summary>Mała podpowiedź</summary>

Schemat: oblicz średnią i odchylenie → `if ...: raise ValueError("Stała kolumna")` → `return ...`. W NumPy domyślne `ddof` różni się od pandas, więc podaj parametr jawnie.

</details>

In [ ]:
# Twoje rozwiązanie


### Kolejne pomysły dla chętnych · poza podstawowymi 120 minutami pracy

**A. Cena względem sąsiadów z segmentu.** Przez `groupby("strefa_metra", observed=True)["cena_twd_m2"].transform("median")` przypisz każdej nieruchomości medianę jej strefy. Dodaj procentową różnicę względem tej mediany. Wybierz rekordy spełniające wymagania klienta i tańsze od mediany własnej strefy. Dlaczego „tanie względem strefy” nie znaczy automatycznie „okazja”?

**B. Symulacja większej odległości.** Za pomocą pętli dla progów 300, 500, 800, 1000 i 1500 metrów policz liczbę kandydatów oraz medianę ceny, utrzymując pozostałe wymagania. Zbierz wyniki w DataFrame. Pokaż, co klient zyskuje przez rozszerzenie kryterium odległości.

**C. Standaryzacja a min–max.** Oblicz cenę po przekształceniu min–max. W jednej tabeli porównaj minimum, maksimum, średnią i odchylenie standardowe ceny w TWD/m², ceny standaryzowanej i min–max. Co zmienia każde przekształcenie?

**D. Funkcja rekomendująca.** Zdefiniuj `wybierz_nieruchomosci(tabela, max_wiek=20, max_metro=800, min_sklepy=3)`. Funkcja ma zwracać tabelę dopasowań posortowaną po cenie i odległości. Sprawdź dwa profile klientów oraz sytuację bez dopasowań.

In [ ]:
# Zadanie dodatkowe — Twoje rozwiązanie


## Ściąga do powtórki

| Cel | Narzędzie |
|---|---|
| Wgląd | `head()`, `info()`, `describe()`, `shape` |
| Braki | `isna()`, `notna()` |
| Maska | `(warunek1) & (warunek2)`, `|`, `~` |
| Lista wartości / przedział | `isin([...])`, `between(a, b)` |
| Wybór | `.loc[maska, lista_kolumn]` |
| Ranking | `sort_values([...])`, `head(n)`, `nlargest(...)` |
| Kategorie przedziałowe | `pd.cut(...)` |
| Porównanie grup | `groupby(...).agg(...)` |
| Tabela przestawna | `pivot_table(...)`, `pd.crosstab(...)` |
| Tekst | `.str.contains()`, `.str.extract()`, `.str.replace()` |
| Daty | `pd.to_datetime(...)`, `pd.Timedelta(...)` |
| Własna funkcja | `def ...:` i `return` |

Materiał rozwija zakres `raport2.Rmd` w Pythonie. Całe podstawowe laboratorium jest oparte na jednym case study nieruchomościowym.